In [1]:
import pandas as pd
import numpy as np 

In [2]:
df = pd.read_csv('../open_splice_experimental_master_unique_sequence_all_predictor_cols.tsv', sep='\t')

In [3]:
df.shape

(590104, 169)

In [4]:
# inspect if 0 or 1 occurs in the dataset
cols = [
    "alphagenome_minigene__alphagenome_minigene_acceptor_mut",
    "alphagenome_minigene__alphagenome_minigene_donor_mut"
]

for col in cols:
    print(
        col,
        df[col].min(),
        df[col].max(),
        (df[col] == 0).sum(),
        (df[col] == 1).sum()
    )

alphagenome_minigene__alphagenome_minigene_acceptor_mut 0.0 0.96875 12941 0
alphagenome_minigene__alphagenome_minigene_donor_mut 0.0 0.96875 13168 0


In [6]:
from scipy.special import logit

eps = 1e-6

acc = df[
    "alphagenome_minigene__alphagenome_minigene_acceptor_mut"
].clip(eps, 1 - eps)

don = df[
    "alphagenome_minigene__alphagenome_minigene_donor_mut"
].clip(eps, 1 - eps)

df["alphag_mut_acceptor_logit"] = logit(acc)
df["alphag_mut_donor_logit"] = logit(don)

df["alphag_mean_logit"] = df[
    ["alphag_mut_acceptor_logit", "alphag_mut_donor_logit"]
].mean(axis=1)

df["alphag_prod_logit"] = logit((acc * don).clip(eps, 1 - eps))

/var/folders/0w/9zhskz553v76zkbgq7pjk6jw0000gn/T/ipykernel_87897/1609244267.py:16: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["alphag_mean_logit"] = df[
/var/folders/0w/9zhskz553v76zkbgq7pjk6jw0000gn/T/ipykernel_87897/1609244267.py:20: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["alphag_prod_logit"] = logit((acc * don).clip(eps, 1 - eps))


In [7]:
df[[
    "alphag_mut_acceptor_logit",
    "alphag_mut_donor_logit",
    "alphag_mean_logit",
    "alphag_prod_logit"
]].describe()

,alphag_mut_acceptor_logit,alphag_mut_donor_logit,alphag_mean_logit,alphag_prod_logit
count,590104.000000,590104.000000,590104.000000,590104.000000
mean,-1.701494,-1.925883,-1.813689,-4.104376
std,2.908924,2.673756,2.460423,3.905340
min,-13.815510,-13.815510,-13.815510,-13.815510
25%,-3.039928,-2.958924,-3.142759,-6.375857
50%,-0.761322,-1.077886,-0.934979,-2.478740
75%,0.267204,-0.331117,-0.050914,-1.208594
max,3.433987,3.433987,3.433987,2.724840


In [10]:
# check splice ai distribution

spliceai_cols = [
    "spliceai_minigene__spliceai_minigene_acceptor_wt",
    "spliceai_minigene__spliceai_minigene_donor_wt",
    "spliceai_minigene__spliceai_minigene_acceptor_mut",
    "spliceai_minigene__spliceai_minigene_donor_mut",
]

for col in spliceai_cols:
    print(
        col,
        "min =", df[col].min(),
        "max =", df[col].max(),
        "zeros =", (df[col] == 0).sum(),
        "ones =", (df[col] == 1).sum(),
    )

spliceai_minigene__spliceai_minigene_acceptor_wt min = 0.0029056123457849 max = 0.9989926218986512 zeros = 0 ones = 0
spliceai_minigene__spliceai_minigene_donor_wt min = 0.008925661444664 max = 0.998262107372284 zeros = 0 ones = 0
spliceai_minigene__spliceai_minigene_acceptor_mut min = 0.0 max = 0.9994634 zeros = 12941 ones = 0
spliceai_minigene__spliceai_minigene_donor_mut min = 0.0 max = 0.9993288 zeros = 13168 ones = 0


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.special import logit
from scipy.stats import linregress, pearsonr, spearmanr


# --------------------------------------------------
# Paths
# --------------------------------------------------

ROOT = Path.cwd().parent if Path.cwd().name == "analysis_script" else Path.cwd()

OUT_DIR = ROOT / "outputs" / "plots" / "alphagenome_mean_prod"
OUT_DIR.mkdir(parents=True, exist_ok=True)


# --------------------------------------------------
# Experimental logit PSI
# --------------------------------------------------

# Experimental PSI is 0-100.
# Keep this clipping, because here it is only to make PSI=0/100 finite on logit scale.
eps = 1e-8

psi_prob = (df["psi"] / 100).clip(eps, 1 - eps)
df["logit_psi"] = logit(psi_prob)


# --------------------------------------------------
# AlphaGenome absolute mutant splice-site scores
# NO clipping here
# --------------------------------------------------

acc_col = "alphagenome_minigene__alphagenome_minigene_acceptor_mut"
don_col = "alphagenome_minigene__alphagenome_minigene_donor_mut"

acc = df[acc_col]
don = df[don_col]

df["alphag_mut_acceptor_logit"] = logit(acc)
df["alphag_mut_donor_logit"] = logit(don)

df["alphag_mean_logit"] = df[
    ["alphag_mut_acceptor_logit", "alphag_mut_donor_logit"]
].mean(axis=1)

df["alphag_prod_logit"] = logit(acc * don)


# --------------------------------------------------
# Helper
# --------------------------------------------------

def get_stats(x, y):
    x = np.asarray(x)
    y = np.asarray(y)

    mask = np.isfinite(x) & np.isfinite(y)

    x_valid = x[mask]
    y_valid = y[mask]

    if len(x_valid) < 2 or np.ptp(x_valid) == 0 or np.ptp(y_valid) == 0:
        return None

    reg = linregress(x_valid, y_valid)
    pearson_r, _ = pearsonr(x_valid, y_valid)
    spearman_r, _ = spearmanr(x_valid, y_valid)

    return {
        "n": len(x_valid),
        "slope": reg.slope,
        "intercept": reg.intercept,
        "pearson": pearson_r,
        "spearman": spearman_r,
        "x": x_valid,
        "y": y_valid,
    }


# --------------------------------------------------
# Plot one figure per exon
# --------------------------------------------------

for exon_id, g in df.groupby("exon_id"):

    g = g.dropna(
        subset=[
            "logit_psi",
            acc_col,
            don_col,
        ]
    ).copy()

    if len(g) < 2:
        continue

    total_n = len(g)

    # Mean-logit is invalid if acceptor OR donor is zero
    mean_zero_mask = (
        (g[acc_col] == 0)
        | (g[don_col] == 0)
    )

    # Product is zero under exactly the same condition
    prod_zero_mask = (
        (g[acc_col] * g[don_col]) == 0
    )

    mean_dropped_pct = 100 * mean_zero_mask.mean()
    prod_dropped_pct = 100 * prod_zero_mask.mean()


    fig, axes = plt.subplots(
        1, 2,
        figsize=(14, 6),
    )

    comparisons = [
        (
            "alphag_mean_logit",
            "AlphaGenome mean splice-site logit",
            "Mean of acceptor/donor logits",
            mean_dropped_pct,
        ),
        (
            "alphag_prod_logit",
            "AlphaGenome product splice-site logit",
            "Logit of acceptor × donor",
            prod_dropped_pct,
        ),
    ]

    for ax, (xcol, xlabel, panel_title, dropped_pct) in zip(
        axes,
        comparisons
    ):

        stats = get_stats(
            g[xcol],
            g["logit_psi"]
        )

        if stats is None:
            ax.set_title(
                f"{panel_title}\n"
                f"Dropped zero-score variants: {dropped_pct:.1f}%"
            )
            continue

        x = stats["x"]
        y = stats["y"]

        ax.scatter(
            x,
            y,
            alpha=0.55,
            s=25
        )

        xline = np.linspace(
            x.min(),
            x.max(),
            200
        )

        yline = (
            stats["slope"] * xline
            + stats["intercept"]
        )

        ax.plot(
            xline,
            yline,
            linewidth=2
        )

        ax.set_title(
            f"{panel_title}\n"
            f"n = {stats['n']} / {total_n}, "
            f"dropped zero-score = {dropped_pct:.1f}%\n"
            f"Slope = {stats['slope']:.3f}, "
            f"Pearson r = {stats['pearson']:.3f}\n"
            f"Spearman ρ = {stats['spearman']:.3f}"
        )

        ax.set_xlabel(xlabel)
        ax.grid(alpha=0.2)

    axes[0].set_ylabel("Measured logit PSI")
    axes[1].set_ylabel("Measured logit PSI")
    axes[1].tick_params(axis="y", labelleft=True)

    fig.suptitle(
        exon_id,
        fontsize=16,
        y=1.02
    )

    plt.tight_layout()

    plt.savefig(
        OUT_DIR / f"{exon_id}_alphagenome_mean_prod_vs_measured.png",
        dpi=200,
        bbox_inches="tight"
    )

    plt.close(fig)


print("Done.")
print("Exons:", df["exon_id"].nunique())
print("Plots saved to:", OUT_DIR)